# Leyendo y escribiendo desde HDFS.

El objetivo de esta parte es leer y escribir archivos desde SPARK sobre HDFS. 

**Nota**: es muy importante que antes de esta parte los servicios de HDFS y YARN estén corriendo en la máquina, puedes hacerlo en este entorno desde una terminal. Tambíen es importante ejecutar la última celda del notebook SparkSQL para terminar la sección anterior. 

In [7]:
# Crear una nueva sesión configurada para HDFS
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PracticaHDFS")
    .master("local[*]")
    .config(
        "spark.hadoop.fs.defaultFS",
        "hdfs://localhost:9000"
    )
    .getOrCreate()
)

In [2]:
# Verificar conexión con HDFS
sc = spark.sparkContext

print("Sistema de archivos predeterminado:",
      sc._jsc.hadoopConfiguration().get("fs.defaultFS"))

Sistema de archivos predeterminado: hdfs://localhost:9000


In [5]:
#Crear Dataframe de ejemplo
datos = [
    (1, "Ana", "Antioquia"),
    (2, "Carlos", "Santander"),
    (3, "Laura", "Cundinamarca")
]

columnas = ["id", "nombre", "departamento"]

df_clientes = spark.createDataFrame(datos, columnas)

In [7]:
#Guardar Datos en HDFS.
ruta = "hdfs:///datos/clientes_parquet"

df_clientes.write \
    .mode("overwrite") \
    .parquet(ruta)

In [8]:
#Leer Datos en HDFS
df_leido = spark.read.parquet(
    "hdfs:///datos/clientes_parquet"
)

df_leido.show()
df_leido.printSchema()


+---+------+------------+
| id|nombre|departamento|
+---+------+------------+
|  2|Carlos|   Santander|
|  3| Laura|Cundinamarca|
|  1|   Ana|   Antioquia|
+---+------+------------+

root
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- departamento: string (nullable = true)



In [9]:
#Agregar consulta SQL. 
df_leido.createOrReplaceTempView("clientes_hdfs")

spark.sql("""
    SELECT departamento, COUNT(*) AS cantidad
    FROM clientes_hdfs
    GROUP BY departamento
""").show()

+------------+--------+
|departamento|cantidad|
+------------+--------+
|Cundinamarca|       1|
|   Santander|       1|
|   Antioquia|       1|
+------------+--------+



In [4]:
#Copiando datos Desde nuestra máquina local de Linux hacia HDFS usando SPARK. 

from py4j.java_gateway import java_import

sc = spark.sparkContext
jvm = sc._jvm
conf = sc._jsc.hadoopConfiguration()

# Sistema de archivos local
fs_local = jvm.org.apache.hadoop.fs.FileSystem.getLocal(conf)

# Sistema de archivos HDFS
uri_hdfs = jvm.java.net.URI("hdfs://localhost:9000")
fs_hdfs = jvm.org.apache.hadoop.fs.FileSystem.get(uri_hdfs, conf)

# Rutas de origen y destino
origen = jvm.org.apache.hadoop.fs.Path("./clientes_1000.csv")
destino = jvm.org.apache.hadoop.fs.Path("/spark_hadoop/clientes_1000.csv")

# Crear el directorio de destino
fs_hdfs.mkdirs(destino.getParent())

# Copiar el archivo
fs_hdfs.copyFromLocalFile(False, origen, destino)

print("Archivo copiado a HDFS")

Archivo copiado a HDFS


# Lanzando trabajos desde YARN. 

Para usar YARN como gestor de recursos requerimos una sección de SPARK que tenga configurado YARN como gestor de recursos en el parámetro --master.

Con el siguiente código podemos correr el pipeline_clientes desde YARN como trabajo de SPARK. 

In [12]:
#Lanzando el trabajo 
import subprocess

comando = [
    "/opt/spark/bin/spark-submit",
    "--master", "yarn",
    "--deploy-mode", "client",
    "--name", "PipelineClientes",
    "pipeline_clientes.py"
]

proceso = subprocess.run(
    comando,
    capture_output=True,
    text=True
)

print(proceso.stdout)

if proceso.returncode != 0:
    print(proceso.stderr)

Esquema de los datos:
root
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- fecha_nacimiento: date (nullable = true)

Cantidad de registros: 1000
Resumen por departamento:
+---------------+-----------------+-------------+-----------+-----------+
|departamento   |cantidad_clientes|edad_promedio|edad_minima|edad_maxima|
+---------------+-----------------+-------------+-----------+-----------+
|Bolivar        |190              |36.79        |17         |56         |
|Atlantico      |168              |38.46        |18         |56         |
|Cundinamarca   |166              |36.05        |17         |56         |
|Santander      |166              |38.73        |18         |56         |
|Antioquia      |160              |37.68        |17         |56         |
|Valle del Cauca|150              |35.19        |18         |56         |
+---

In [14]:
# Crear una nueva sesión configurada para HDFS ya que el trabajo anterior dio por finalizada cualquier sesión activa. 

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PracticaHDFS")
    .master("local[*]")
    .config(
        "spark.hadoop.fs.defaultFS",
        "hdfs://localhost:9000"
    )
    .getOrCreate()
)
ruta_salida = "hdfs:///spark_hadoop/salidas/resumen_clientes_parquet"

resultado = spark.read.parquet(ruta_salida)

resultado.show(truncate=False)

# Detener la sesión actual
spark.stop()

+---------------+-----------------+-------------+-----------+-----------+
|departamento   |cantidad_clientes|edad_promedio|edad_minima|edad_maxima|
+---------------+-----------------+-------------+-----------+-----------+
|Bolivar        |190              |36.79        |17         |56         |
|Atlantico      |168              |38.46        |18         |56         |
|Cundinamarca   |166              |36.05        |17         |56         |
|Santander      |166              |38.73        |18         |56         |
|Antioquia      |160              |37.68        |17         |56         |
|Valle del Cauca|150              |35.19        |18         |56         |
+---------------+-----------------+-------------+-----------+-----------+



## Ejercicio Conteo Palabras desde Spark


In [16]:
#Copiando datos Desde nuestra máquina local de Linux hacia HDFS usando SPARK. 

# Crear una nueva sesión configurada para HDFS
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PracticaHDFS")
    .master("local[*]")
    .config(
        "spark.hadoop.fs.defaultFS",
        "hdfs://localhost:9000"
    )
    .getOrCreate()
)

from py4j.java_gateway import java_import

sc = spark.sparkContext
jvm = sc._jvm
conf = sc._jsc.hadoopConfiguration()

# Sistema de archivos local
fs_local = jvm.org.apache.hadoop.fs.FileSystem.getLocal(conf)

# Sistema de archivos HDFS
uri_hdfs = jvm.java.net.URI("hdfs://localhost:9000")
fs_hdfs = jvm.org.apache.hadoop.fs.FileSystem.get(uri_hdfs, conf)

# Rutas de origen y destino
origen = jvm.org.apache.hadoop.fs.Path("./advs.txt")
destino = jvm.org.apache.hadoop.fs.Path("/spark_hadoop/palabras.txt")

# Crear el directorio de destino
fs_hdfs.mkdirs(destino.getParent())

# Copiar el archivo
fs_hdfs.copyFromLocalFile(False, origen, destino)


# Detener la sesión actual
spark.stop()
print("Archivo copiado a HDFS")

Archivo copiado a HDFS


In [17]:
# Ejecutar el pipeline de WordCount.py

import subprocess
import os

comando = [
    "/opt/spark/bin/spark-submit",
    "--master", "yarn",
    "--deploy-mode", "client",
    "--name", "PipelineWordCount",
    "pipeline_WordCount.py"
]

proceso = subprocess.run(
    comando,
    capture_output=True,
    text=True,
    cwd=os.getcwd()
)

print(proceso.stdout)

if proceso.returncode != 0:
    print("Error al ejecutar el pipeline:")
    print(proceso.stderr)
else:
    print("Trabajo finalizado correctamente.")


Resultado del WordCount:
+-------+----------+
|palabra|frecuencia|
+-------+----------+
|the    |5621      |
|and    |3002      |
|i      |2994      |
|to     |2685      |
|of     |2658      |
|a      |2626      |
|in     |1758      |
|that   |1735      |
|it     |1697      |
|you    |1477      |
|he     |1463      |
|was    |1409      |
|his    |1159      |
|is     |1123      |
|my     |1006      |
|have   |922       |
|as     |852       |
|with   |830       |
|had    |829       |
|which  |769       |
|at     |767       |
|for    |727       |
|but    |648       |
|not    |636       |
|me     |634       |
|be     |628       |
|we     |523       |
|there  |500       |
|from   |498       |
|this   |490       |
|said   |486       |
|upon   |464       |
|holmes |461       |
|so     |446       |
|him    |433       |
|her    |429       |
|she    |425       |
|very   |400       |
|your   |395       |
|been   |393       |
|no     |389       |
|all    |386       |
|what   |386       |
|on     |

In [20]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ConsultarWordCount")
    .master("local[*]")
    .config(
        "spark.hadoop.fs.defaultFS",
        "hdfs://localhost:9000"
    )
    .getOrCreate()
)

resultado = spark.read.parquet(
    "hdfs:///spark_hadoop/salidas/wordcount_parquet"
)

resultado.orderBy("frecuencia", ascending=False).show(50)

+-------+----------+
|palabra|frecuencia|
+-------+----------+
|    the|      5621|
|    and|      3002|
|      i|      2994|
|     to|      2685|
|     of|      2658|
|      a|      2626|
|     in|      1758|
|   that|      1735|
|     it|      1697|
|    you|      1477|
|     he|      1463|
|    was|      1409|
|    his|      1159|
|     is|      1123|
|     my|      1006|
|   have|       922|
|     as|       852|
|   with|       830|
|    had|       829|
|  which|       769|
|     at|       767|
|    for|       727|
|    but|       648|
|    not|       636|
|     me|       634|
|     be|       628|
|     we|       523|
|  there|       500|
|   from|       498|
|   this|       490|
|   said|       486|
|   upon|       464|
| holmes|       461|
|     so|       446|
|    him|       433|
|    her|       429|
|    she|       425|
|   very|       400|
|   your|       395|
|   been|       393|
|     no|       389|
|    all|       386|
|   what|       386|
|     on|       381|
|    one|    